## Importing libraries and connection establishment 

In [1]:
import duckdb
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [2]:
con = duckdb.connect("../aact.duckdb", read_only=True)

def q(sql):
    cur = con.cursor()
    try:
        return cur.execute(sql).df()
    finally:
        cur.close()

In [3]:
tables = ["studies", "calculated_values", "designs", "eligibilities",
          "sponsors", "browse_conditions", "interventions", "countries"]


In [4]:
for t in tables:
    n = con.execute(f"select count(*) from raw.{t}").fetchone()[0]
    print(f"{t:20s} {n:>10,}")

studies                 610,749
calculated_values       610,749
designs                 605,944
eligibilities           609,760
sponsors                973,029
browse_conditions     4,422,890
interventions         1,033,280
countries               826,206


In [5]:
for t in tables:
    rows, trials = con.execute(
        f"select count(*), count(distinct nct_id) from raw.{t}"
    ).fetchone()
    print(f"{t:20s} rows={rows:>10,}  trials={trials:>10,}  rows/trial={rows/trials:.2f}")

studies              rows=   610,749  trials=   610,749  rows/trial=1.00
calculated_values    rows=   610,749  trials=   610,749  rows/trial=1.00
designs              rows=   605,944  trials=   605,944  rows/trial=1.00
eligibilities        rows=   609,760  trials=   609,760  rows/trial=1.00
sponsors             rows=   973,029  trials=   610,749  rows/trial=1.59
browse_conditions    rows= 4,422,890  trials=   481,143  rows/trial=9.19
interventions        rows= 1,033,280  trials=   548,718  rows/trial=1.88
countries            rows=   826,206  trials=   555,864  rows/trial=1.49


# creating dataframe from sql

## Studies tables

In [6]:
studies = q("""
select nct_id, enrollment, enrollment_type, overall_status, phase, study_type,
       start_date_type, start_date, completion_date_type, completion_date,
       number_of_arms, source, has_dmc
from raw.studies
""")
studies.head(5)

,nct_id,enrollment,enrollment_type,overall_status,phase,study_type,start_date_type,start_date,completion_date_type,completion_date,number_of_arms,source,has_dmc
0,NCT03668340,49,ACTUAL,ACTIVE_NOT_RECRUITING,PHASE2,INTERVENTIONAL,ACTUAL,2018-10-22,ESTIMATED,2027-09-01,3,Dana-Farber Cancer Institute,True
1,NCT05894070,100,ESTIMATED,RECRUITING,NaN,OBSERVATIONAL,ACTUAL,2023-08-02,ESTIMATED,2026-06-30,<NA>,"Insel Gruppe AG, University Hospital Bern",False
2,NCT05344599,70,ACTUAL,ACTIVE_NOT_RECRUITING,NaN,OBSERVATIONAL,ACTUAL,2022-01-19,ESTIMATED,2026-12-30,<NA>,Vanderbilt University Medical Center,True
3,NCT02880306,741379,ACTUAL,COMPLETED,NaN,OBSERVATIONAL,NaN,2014-07-31,ACTUAL,2016-08-31,<NA>,"Taipei Tzu Chi Hospital, Buddhist Tzu Chi Medi...",True
4,NCT04308356,71,ACTUAL,COMPLETED,NaN,OBSERVATIONAL,ACTUAL,2020-06-03,ACTUAL,2024-07-07,<NA>,Assistance Publique - Hôpitaux de Paris,False


### Checking for Null values

In [9]:
studies.isnull().sum()

nct_id                       0
enrollment                7147
enrollment_type          17145
overall_status               0
phase                   144819
study_type                 989
start_date_type         186145
start_date                5379
completion_date_type     23427
completion_date          16694
number_of_arms          168639
source                       0
has_dmc                 108315
dtype: int64

### Exploring the overall_status column since we have to create the target column using the attributes from it.

Shall be creating a new feature called "at_risk" where we shall store 
    1. For trails terminated or withdrawn => 1
    2. For trails completed => 0

In [11]:
studies['overall_status'].shape

(610749,)

In [8]:
studies['overall_status'].value_counts()

overall_status
COMPLETED                    334323
UNKNOWN                       98873
RECRUITING                    64730
TERMINATED                    34685
NOT_YET_RECRUITING            30012
ACTIVE_NOT_RECRUITING         22074
WITHDRAWN                     16972
ENROLLING_BY_INVITATION        5245
SUSPENDED                      1770
WITHHELD                        989
NO_LONGER_AVAILABLE             544
AVAILABLE                       250
APPROVED_FOR_MARKETING          245
TEMPORARILY_NOT_AVAILABLE        37
Name: count, dtype: int64

In [25]:
studies[
    studies['overall_status'].isin(['COMPLETED', 'TERMINATED', 'WITHDRAWN'])
].value_counts()

nct_id       enrollment  enrollment_type  overall_status  phase          study_type      start_date_type  start_date  completion_date_type  completion_date  number_of_arms  source                                  has_dmc
NCT01902381  12          ACTUAL           TERMINATED      PHASE2         INTERVENTIONAL  ACTUAL           2013-08-31  ACTUAL                2018-11-26       1               Wake Forest University Health Sciences  False      1
NCT02755376  30          ACTUAL           COMPLETED       NA             INTERVENTIONAL  ACTUAL           2014-01-01  ACTUAL                2018-12-01       3               Samsung Medical Center                  False      1
NCT04871295  370         ACTUAL           COMPLETED       NA             INTERVENTIONAL  ACTUAL           2021-04-15  ACTUAL                2022-02-15       2               Noom Inc.                               False      1
NCT03087201  98          ACTUAL           COMPLETED       PHASE3         INTERVENTIONAL  ACTUAL      

In [24]:
filtered_studies

,nct_id,enrollment,enrollment_type,overall_status,phase,study_type,start_date_type,start_date,completion_date_type,completion_date,number_of_arms,source,has_dmc
3,NCT02880306,741379,ACTUAL,COMPLETED,NaN,OBSERVATIONAL,NaN,2014-07-31,ACTUAL,2016-08-31,<NA>,"Taipei Tzu Chi Hospital, Buddhist Tzu Chi Medi...",True
4,NCT04308356,71,ACTUAL,COMPLETED,NaN,OBSERVATIONAL,ACTUAL,2020-06-03,ACTUAL,2024-07-07,<NA>,Assistance Publique - Hôpitaux de Paris,False
5,NCT05048368,0,ACTUAL,WITHDRAWN,PHASE1,INTERVENTIONAL,ESTIMATED,2026-10-30,ESTIMATED,2026-12-30,4,"Sunshine Lake Pharma Co., Ltd.",<NA>
6,NCT01881152,1772,ACTUAL,COMPLETED,NA,INTERVENTIONAL,NaN,2013-06-30,ACTUAL,2014-12-31,2,Azienda Ospedaliero-Universitaria di Parma,False
7,NCT00197106,176,ACTUAL,COMPLETED,PHASE4,INTERVENTIONAL,NaN,2005-06-30,ACTUAL,2008-10-31,2,GlaxoSmithKline,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...
610742,NCT00555841,27,ACTUAL,TERMINATED,PHASE2,INTERVENTIONAL,NaN,2007-03-31,ACTUAL,2008-09-30,2,"Sigma-Tau Research, Inc.",True
610744,NCT01038297,30,ACTUAL,COMPLETED,PHASE2,INTERVENTIONAL,ACTUAL,2009-12-01,ACTUAL,2010-07-07,2,Pfizer,False
610746,NCT02012114,65,ACTUAL,COMPLETED,NA,INTERVENTIONAL,NaN,2011-12-31,ACTUAL,2021-12-14,1,Hospices Civils de Lyon,True
610747,NCT05127837,601,ACTUAL,COMPLETED,NA,INTERVENTIONAL,ACTUAL,2022-06-28,ACTUAL,2024-12-09,2,University of Washington,True


<b> <u> Note:</u> we need to use only the rows where the overall_status is completed, terminated or withdrawn </b>

## calculated_values table 

In [26]:
calculated_values = q("select nct_id, number_of_facilities, registered_in_calendar_year,minimum_age_num, maximum_age_num, has_us_facility, has_single_facility from raw.calculated_values")
calculated_values.head(5)

,nct_id,number_of_facilities,registered_in_calendar_year,minimum_age_num,maximum_age_num,has_us_facility,has_single_facility
0,NCT03572205,1,2018,50,65,False,True
1,NCT03116581,1,2017,18,35,False,True
2,NCT00900887,1,2009,40,80,False,True
3,NCT03050775,0,2017,18,<NA>,<NA>,<NA>
4,NCT04600817,16,2020,18,70,False,False


## Design table

In [28]:
designs = q("select nct_id, allocation, intervention_model, primary_purpose, masking, observational_model from raw.designs")
designs.head(5)

,nct_id,allocation,intervention_model,primary_purpose,masking,observational_model
0,NCT06051110,NON_RANDOMIZED,PARALLEL,DIAGNOSTIC,NONE,NaN
1,NCT05084599,RANDOMIZED,PARALLEL,PREVENTION,NONE,NaN
2,NCT05390996,RANDOMIZED,PARALLEL,TREATMENT,NONE,NaN
3,NCT00170274,RANDOMIZED,PARALLEL,TREATMENT,NONE,NaN
4,NCT02986750,RANDOMIZED,PARALLEL,BASIC_SCIENCE,SINGLE,NaN


## eligibilities

In [29]:
eligibilities = q("select nct_id, gender, minimum_age, maximum_age, healthy_volunteers from raw.eligibilities")
eligibilities1 = q("select nct_id, gender, healthy_volunteers, sampling_method, criteria from raw.eligibilities")
print(eligibilities.head(5))
print(eligibilities1.head(5))

        nct_id gender minimum_age maximum_age  healthy_volunteers
0  NCT03621358    ALL    18 Years    55 Years                True
1  NCT04198636    ALL    18 Years    50 Years                True
2  NCT03744910    ALL    18 Years    75 Years               False
3  NCT03358992    ALL         NaN         NaN               False
4  NCT03781609    ALL    18 Years    60 Years               False
        nct_id gender  healthy_volunteers         sampling_method                                           criteria
0  NCT03621358    ALL                True                     NaN  Inclusion Criteria:\n\n* Men and women from 18...
1  NCT04198636    ALL                True                     NaN  Inclusion Criteria:\n\n1. Sign the informed co...
2  NCT03744910    ALL               False                     NaN  * Inclusion criteria:\n\n  1. Age 18-75 years....
3  NCT03358992    ALL               False  NON_PROBABILITY_SAMPLE  Inclusion Criteria:\n\n* 1\) histologically co...
4  NCT03781609    A

In [30]:
eligibilities['minimum_age'].value_counts()

minimum_age
18 Years     382183
20 Years      22500
40 Years      12763
21 Years      11337
50 Years       9777
              ...  
57 Days           1
17 Weeks          1
60 Weeks          1
112 Days          1
52 Months         1
Name: count, Length: 336, dtype: int64

## Sponsers

In [31]:
calculated_values = q("select nct_id, number_of_facilities, registered_in_calendar_year,minimum_age_num, maximum_age_num, minimum_age_unit ,has_us_facility, has_single_facility from raw.calculated_values")
calculated_values.head(5)

,nct_id,number_of_facilities,registered_in_calendar_year,minimum_age_num,maximum_age_num,minimum_age_unit,has_us_facility,has_single_facility
0,NCT03572205,1,2018,50,65,year,False,True
1,NCT03116581,1,2017,18,35,year,False,True
2,NCT00900887,1,2009,40,80,year,False,True
3,NCT03050775,0,2017,18,<NA>,year,<NA>,<NA>
4,NCT04600817,16,2020,18,70,year,False,False


In [32]:
calculated_values['minimum_age_unit'].value_counts()

minimum_age_unit
year      555276
month       9153
day         3647
week        2050
hour         641
minute       464
Name: count, dtype: int64

## Browse_condition

## Interventions

## countries